# FisheriesAudit ALG 2026 — Entrega #04
## Contexto internacional: Argentina en la pesca mundial (FAO FIRMS)

**Autor:** Ariel L. Giamportone  
**Filiación:** Ingeniero Pesquero | Docente Investigador | Data Scientist  
**Serie:** FisheriesAudit ALG 2026 — Gobernanza Pesquera Argentina  
**Fecha:** 2026-05-31

---

### Resumen

Este análisis ubica a Argentina en el contexto de la pesca mundial utilizando datos públicos
de la FAO (Food and Agriculture Organization). Para las principales especies del Mar Argentino,
se examina: (1) la participación de Argentina en las capturas globales del Área FAO 41
(Atlántico Sudoccidental); (2) el estado de los stocks según la evaluación FAO FIRMS;
y (3) la coherencia entre las decisiones del CFP (Consejo Federal Pesquero) y el contexto
internacional de sostenibilidad.

El análisis constituye la cuarta entrega de la Serie FisheriesAudit ALG y cierra el
**Triángulo de Contexto**: datos locales (CBA→CMP, Entregas #01–#03) + contexto global (FAO).

**Palabras clave:** FAO FIRMS, capturas globales, estado de stocks, Área 41,
Atlántico Sudoccidental, sostenibilidad pesquera, Argentina

---

### Marco de referencia

> FAO (2022) *The State of World Fisheries and Aquaculture*: el 34.2% de los stocks
> pesqueros mundiales están explotados a niveles biológicamente insostenibles.
> Los stocks del Atlántico Sudoccidental (Área 41) incluyen algunas de las especies
> más valiosas de la Argentina y presentan patrones de explotación que merecen
> atención desde la política pública.

## Setup — Importaciones y configuración

In [ ]:
%matplotlib inline
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path(".").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src.acquisition.fao_firms_scraper import (
    FAOFIRMSScraper,
    calcular_share_argentina,
    estado_stock_label,
    estado_stock_color,
    ESPECIE_FAO_CODES,
)
from src.analysis.research_exporter import (
    FAOExporter,
    SERIES_BRAND,
    SERIES_NAME,
    TestResult,
)
from src.analysis.linkedin_formatter import (
    LinkedInPost,
    HASHTAGS_PERSONAL,
    HASHTAGS_PESQUEROS_IA,
    SERIE_HEADER,
)

DB_PATH = Path("data/processed/catalog.db")
OUT_DIR = Path("outputs/FisheriesAudit_ALG")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figuras").mkdir(exist_ok=True)

print("Dependencias cargadas")
print(f"  pandas {pd.__version__} | scipy {stats.__version__} | matplotlib {plt.matplotlib.__version__}")
print(f"  DB: {DB_PATH} | Output: {OUT_DIR}")

## 1. Estado del corpus FAO

Verificamos los datos disponibles en el catálogo local (SQLite `catalog.db`).
Si las tablas están vacías, cargamos los datos semilla verificados de FAO FishStat
y FAO FIRMS para el Área 41 (Atlántico Sudoccidental).

In [ ]:
scraper = FAOFIRMSScraper(DB_PATH)

# Inicializar schema y cargar datos seed si es necesario
n_cap, n_status = scraper.seed_data()
if n_cap > 0 or n_status > 0:
    print(f"Datos seed cargados: {n_cap} registros de capturas, {n_status} de estado de stocks")

df_cap = scraper.get_capturas_df()
df_status = scraper.get_stock_status_df()

print("\n=== Corpus FAO disponible ===")
print(f"  Capturas FAO (fao_capturas):   {len(df_cap)} registros")
print(f"  Estado stocks (fao_stock_status): {len(df_status)} registros")

if not df_cap.empty:
    print(f"\n  Especies con datos de captura:")
    for code, grp in df_cap.groupby("especie_fao_code"):
        paises = sorted(grp["pais"].unique())
        years = sorted(grp["year"].unique())
        print(f"    {code}: años {years[0]}–{years[-1]}, países: {paises}")

if not df_status.empty:
    print(f"\n  Stocks evaluados:")
    for _, row in df_status.iterrows():
        print(f"    {row['especie_fao_code']}: {row['estado_stock_desc']} "
              f"({row.get('tendencia', '?')}) — eval. {row.get('year_evaluacion', '?')}")

### Catálogo de especies y códigos FAO

Las 8 especies principales del Mar Argentino con sus códigos FAO ASFIS
y recursos FIRMS del Área 41.

In [ ]:
df_esp = pd.DataFrame([
    {
        "especie_local": k,
        "código_fao": v["fao_code"],
        "nombre_fao": v["nombre_fao"],
        "nombre_científico": v["nombre_cientifico"],
        "recurso_firms": v["firms_resource"],
    }
    for k, v in ESPECIE_FAO_CODES.items()
])

print("Especies del Mar Argentino — Catálogo FAO ASFIS")
print(df_esp.to_string(index=False))

## 2. Capturas históricas: Argentina en el Atlántico SW y mundo

Visualizamos las capturas argentinas por especie y comparamos con el total del área.
Los datos provienen de FAO FishStat (series oficiales verificadas).

**Fuente:** FAO (2024). *Global Capture Production 1950–2022*. FishStat J.

In [ ]:
if not df_cap.empty:
    # Tabla resumen: captura argentina por especie y año
    pivot = df_cap[df_cap["pais"] == "Argentina"].pivot_table(
        index="especie_fao_code",
        columns="year",
        values="captura_tn",
        aggfunc="sum",
    )
    print("Capturas Argentina (toneladas) — Área FAO 41")
    print("=" * 60)
    print(pivot.to_string(float_format=lambda x: f"{x:,.0f}"))

    print("\nCapturas totales Área FAO 41 (toneladas)")
    print("=" * 60)
    pivot_total = df_cap[df_cap["pais"] == "Total"].pivot_table(
        index="especie_fao_code",
        columns="year",
        values="captura_tn",
        aggfunc="sum",
    )
    print(pivot_total.to_string(float_format=lambda x: f"{x:,.0f}"))
else:
    print("Sin datos de capturas FAO disponibles.")
    print("Ejecutá scraper.seed_data() para cargar datos semilla.")

## 3. Share de Argentina en capturas mundiales

**Figura 1.** Participación porcentual de Argentina en la captura total del Área FAO 41
para cada especie (último año con datos disponibles).

El color de cada barra refleja el estado del stock según FAO FIRMS:
- Verde = Plena explotación (F)
- Amarillo = Moderada explotación / En recuperación (U/R)
- Rojo = Sobrexplotado (O)
- Rojo oscuro = Agotado (D)

In [ ]:
fexp = FAOExporter(scraper, output_dir=OUT_DIR)

fig1 = fexp.figura_share_argentina_historico(save=True)
plt.show()
print(f"Figura guardada: {OUT_DIR}/figuras/share_argentina_historico.png")

In [ ]:
# Tabla: share% por especie
if not df_cap.empty:
    df_arg_share = calcular_share_argentina(df_cap)
    if not df_arg_share.empty and "share_arg_pct" in df_arg_share.columns:
        df_share_latest = (
            df_arg_share.dropna(subset=["share_arg_pct"])
            .sort_values("year")
            .groupby("especie_fao_code")
            .last()[["especie", "year", "captura_tn", "share_arg_pct"]]
            .reset_index()
        )
        print("Share de Argentina en captura total Área FAO 41 (último año disponible)")
        print("=" * 70)
        for _, row in df_share_latest.sort_values("share_arg_pct", ascending=False).iterrows():
            print(
                f"  {row['especie_fao_code']} ({row.get('especie', ''):<20}) "
                f"Año: {int(row['year'])}  "
                f"Argentina: {row['captura_tn']:>10,.0f} t  "
                f"Share: {row['share_arg_pct']:>6.1f}%"
            )
    else:
        print("No se pudo calcular share%. Se requieren datos de 'Total' Área 41.")
else:
    print("Sin datos de capturas.")

## 4. Estado de los stocks — Semáforo FAO

**Figura 2.** Estado de los stocks pesqueros argentinos según la evaluación FAO FIRMS
para el Área 41 (Atlántico Sudoccidental).

| Código | Descripción | Implicancia para la gestión |
|--------|-------------|-----------------------------|
| F | Plena explotación | En límite sostenible — monitoreo continuo |
| O | Sobrexplotado | Por encima del RMS — requiere reducción de capturas |
| U | Subexplotado | Margen para aumentar capturas sosteniblemente |
| R | En recuperación | Mejoría detectada — mantener medidas de conservación |
| D | Agotado | Colapso del stock — requiere veda o restricción severa |

In [ ]:
fig2 = fexp.figura_estado_stocks_mundo(save=True)
plt.show()
print(f"Figura guardada: {OUT_DIR}/figuras/estado_stocks_mundo.png")

In [ ]:
# Tabla resumen de estados de stock
if not df_status.empty:
    print("Estado de stocks FAO FIRMS — Área 41 (Atlántico Sudoccidental)")
    print("=" * 70)
    for _, row in df_status.iterrows():
        estado = row.get("estado_stock", "?")
        label = estado_stock_label(str(estado))
        tendencia = row.get("tendencia", "desconocida")
        year_eval = row.get("year_evaluacion", "?")
        notas = str(row.get("notas", ""))[:80] + "..." if len(str(row.get("notas", ""))) > 80 else row.get("notas", "")
        print(f"\n  {row.get('especie_fao_code')} — {row.get('especie', '')}")
        print(f"    Estado: {estado} ({label}) | Tendencia: {tendencia} | Año eval.: {year_eval}")
        print(f"    Nota: {notas}")

    n_sobre = (df_status["estado_stock"].isin(["O", "D"])).sum()
    n_total = len(df_status)
    print(f"\n  RESUMEN: {n_sobre}/{n_total} stocks sobrexplotados o agotados "
          f"({n_sobre/n_total*100:.1f}%)")
    print(f"  Referencia FAO global (2022): 34.2% sobrexplotados")
else:
    print("Sin datos de estado de stocks disponibles.")

## 5. Análisis por especie: merluza (HKP) en contexto mundial

**Figura 3.** Capturas de merluza hubbsi (*Merluccius hubbsi*) en el Área FAO 41:
Argentina vs total del área.

La merluza común argentina es la especie comercial más importante del Mar Argentino
y representa la casi totalidad de las capturas de la especie en el Atlántico Sudoccidental.
Su stock norte (al norte de 41°S) está evaluado como sobrexplotado por FAO FIRMS/INIDEP.

In [ ]:
fig3 = fexp.figura_capturas_argentina_vs_mundo("HKP", save=True)
plt.show()
print(f"Figura guardada: {OUT_DIR}/figuras/capturas_fao_hkp.png")

In [ ]:
# Contexto completo para HKP
ctx_hkp = scraper.get_contexto_especie("HKP")
print("Contexto internacional — Merluza hubbsi (HKP)")
print("=" * 50)
for k, v in ctx_hkp.items():
    if v is not None:
        print(f"  {k:35s}: {v}")

print()

# Otros contextos disponibles
for code in ["SNA", "SQA", "TOP"]:
    ctx = scraper.get_contexto_especie(code)
    if ctx and ctx.get("captura_argentina_tn"):
        share = ctx.get("share_argentina_pct", "N/A")
        estado = ctx.get("estado_stock_desc", "N/A")
        print(f"  {code}: {ctx.get('captura_argentina_tn', 0):>10,.0f} t "
              f"| share {share}% | estado: {estado}")

## 6. Tendencias: ¿Argentina sube o baja?

Aplicamos el test de Kendall tau para detectar tendencias monótonas en las capturas
argentinas a lo largo del tiempo, por especie.

**H₀:** No hay tendencia monótona en las capturas argentinas.  
**H₁ (bilateral):** Las capturas muestran tendencia creciente o decreciente.  
**α = 0.05**

In [ ]:
print("Test de Kendall tau — Tendencia en capturas argentinas por especie")
print("=" * 70)

resultados_tendencia = []
codigos_con_datos = df_cap["especie_fao_code"].unique() if not df_cap.empty else []

for code in codigos_con_datos:
    tr = fexp.test_tendencia_captura_argentina(code)
    resultados_tendencia.append(tr)
    sig = "*** p<0.05" if tr.significativo else f"    p={tr.p_value:.3f}"
    tau_str = f"τ={tr.estadistico:.3f}" if not np.isnan(tr.estadistico) else "τ=N/A"
    print(f"\n  [{sig:12s}] {tr.nombre}")
    print(f"   {tau_str} | n={tr.n}")
    print(f"   {tr.interpretacion}")

if not codigos_con_datos.size if hasattr(codigos_con_datos, 'size') else not list(codigos_con_datos):
    # Prueba con HKP como ejemplo aun sin datos suficientes
    tr = fexp.test_tendencia_captura_argentina("HKP")
    print(f"  [HKP] {tr.interpretacion}")

## 7. Sobrexplotación global: ¿cuántas especies argentinas están en zona de riesgo?

Comparamos la proporción de stocks sobrexplotados en Argentina contra el promedio
global reportado por FAO (34.2% en 2022).

**Test:** Binomial exacto de una muestra.  
**H₀:** La proporción de sobrexplotación en Argentina ≤ promedio global FAO (34.2%).  
**H₁ (unilateral):** Argentina supera el promedio global.  
**α = 0.05**

*Referencia:* FAO (2022). *The State of World Fisheries and Aquaculture 2022.*
Rome. Licence: CC BY-NC-SA 3.0 IGO.

In [ ]:
test_sobrex = fexp.test_sobrexplotacion_global()

print("Test de proporciones — Sobrexplotación Argentina vs FAO global")
print("=" * 60)
print(f"  Test: {test_sobrex.nombre}")
print(f"  n stocks analizados : {test_sobrex.n}")
if not np.isnan(test_sobrex.estadistico):
    print(f"  Prop. Argentina     : {test_sobrex.estadistico*100:.1f}%")
print(f"  p-valor             : {test_sobrex.p_value:.4f}")
print(f"  Significativo (a=0.05): {'SÍ' if test_sobrex.significativo else 'NO'}")
print()
print(f"  Interpretación:")
print(f"  {test_sobrex.interpretacion}")

## 8. Triangulación: CBA–CMP vs estado stock FAO
### ¿Coherencia entre política local y contexto global?

Esta sección integra los resultados de las entregas anteriores con el contexto FAO.

El **Triángulo de Contexto Ampliado** cruza:

| Fuente | Indicador | Entrega |
|--------|-----------|--------|
| INIDEP | CBA (Captura Biológicamente Aceptable) | #01 |
| CFP | CMP (Captura Máxima Permisible) | #01 |
| SAGPyA/SIPA | Captura real declarada | #01 |
| FAO FIRMS | Estado del stock global | **#04** |

La pregunta central: **¿cuando FAO dice que un stock está sobrexplotado,
el CFP aprueba cuotas que respetan la CBA de INIDEP?**

In [ ]:
# Triangulación cualitativa: estado FAO vs comportamiento CFP histórico conocido
triangulacion = [
    {
        "especie": "merluza hubbsi",
        "codigo": "HKP",
        "estado_fao": "O — Sobrexplotado",
        "cmp_vs_cba_pattern": "CMP > CBA en años recientes (datos INIDEP ITO 36/2024)",
        "coherencia": "INCONGRUENTE",
        "nota": "Stock Norte sobrexplotado; CFP aprobó cuotas por encima de CBA INIDEP",
    },
    {
        "especie": "langostino",
        "codigo": "SNA",
        "estado_fao": "F — Plena explotación",
        "cmp_vs_cba_pattern": "CMP ~ CBA (monitoreo adecuado)",
        "coherencia": "COHERENTE",
        "nota": "Argentina es productor casi exclusivo; gestión relativamente ajustada",
    },
    {
        "especie": "calamar illex",
        "codigo": "SQA",
        "estado_fao": "F — Plena explotación",
        "cmp_vs_cba_pattern": "Variable; pesca INDNR en alta mar como factor externo",
        "coherencia": "PARCIAL",
        "nota": "Factor externo: flotas china y coreana fuera de ZEE sin regulación CFP",
    },
    {
        "especie": "merluza negra",
        "codigo": "TOP",
        "estado_fao": "F — Plena explotación",
        "cmp_vs_cba_pattern": "Regulada por CCAMLR; CFP sigue TAC internacional",
        "coherencia": "COHERENTE",
        "nota": "Ejemplo de governance internacional efectivo (CCAMLR)",
    },
    {
        "especie": "abadejo",
        "codigo": "POA",
        "estado_fao": "O — Sobrexplotado",
        "cmp_vs_cba_pattern": "INIDEP recomienda reducción; historial de sobreasignación",
        "coherencia": "INCONGRUENTE",
        "nota": "Biomasa desovante reducida; CFP ha aprobado cuotas sobre CBA recomendada",
    },
]

print("Triangulación CBA–CMP–FAO por especie")
print("=" * 70)
for item in triangulacion:
    coherencia_icon = {
        "INCONGRUENTE": "[!]",
        "COHERENTE": "[OK]",
        "PARCIAL": "[~]",
    }.get(item["coherencia"], "[?]")
    print(f"\n{coherencia_icon} {item['especie'].upper()} ({item['codigo']})")
    print(f"   Estado FAO     : {item['estado_fao']}")
    print(f"   Patrón CMP/CBA : {item['cmp_vs_cba_pattern']}")
    print(f"   Coherencia     : {item['coherencia']}")
    print(f"   Nota           : {item['nota']}")

n_incong = sum(1 for x in triangulacion if x["coherencia"] == "INCONGRUENTE")
print(f"\nResumen: {n_incong}/{len(triangulacion)} especies con incongruencia "
      f"entre estado FAO y patrón CFP")

### Figura 4. Diagrama de coherencia — FAO vs CFP

In [ ]:
# Visualización de coherencia FAO vs CFP
fig4, ax = plt.subplots(figsize=(10, 5))

if triangulacion:
    colores_coher = {
        "COHERENTE": "#4CAF50",
        "PARCIAL": "#FFC107",
        "INCONGRUENTE": "#E53935",
    }

    especies = [t["especie"] for t in triangulacion]
    coherencias = [t["coherencia"] for t in triangulacion]
    colores = [colores_coher.get(c, "#9E9E9E") for c in coherencias]

    # Mapa numérico: 0=coherente, 1=parcial, 2=incongruente
    coh_num = {"COHERENTE": 0, "PARCIAL": 1, "INCONGRUENTE": 2}
    vals = [coh_num[c] for c in coherencias]

    y_pos = np.arange(len(especies))
    bars = ax.barh(y_pos, [1] * len(especies), color=colores, alpha=0.85, height=0.6, edgecolor="white")

    for i, (barra, etiq) in enumerate(zip(bars, coherencias)):
        ax.text(
            0.5, barra.get_y() + barra.get_height() / 2,
            etiq, va="center", ha="center", fontsize=10, fontweight="bold", color="white",
        )

    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"{t['especie']} ({t['codigo']})" for t in triangulacion])
    ax.set_xticks([])
    ax.set_title(
        "Coherencia entre estado FAO y política CFP por especie\n"
        "Verde=Coherente · Amarillo=Parcial · Rojo=Incongruente"
    )

    from matplotlib.patches import Patch
    leyenda = [
        Patch(facecolor="#4CAF50", label="Coherente (CFP respeta límites FAO)"),
        Patch(facecolor="#FFC107", label="Parcial (factores externos / mixto)"),
        Patch(facecolor="#E53935", label="Incongruente (CFP supera límites FAO)"),
    ]
    ax.legend(handles=leyenda, loc="lower right", fontsize=8)
else:
    ax.text(
        0.5, 0.5, "Sin datos disponibles para triangulación.",
        ha="center", va="center", transform=ax.transAxes, fontsize=11, color="gray",
    )

fig4.text(0.99, 0.01, SERIES_BRAND, ha="right", va="bottom", fontsize=7, color="gray")
fig4.tight_layout()

fig4.savefig(OUT_DIR / "figuras" / "coherencia_fao_cfp.png", dpi=300, bbox_inches="tight")
fig4.savefig(OUT_DIR / "figuras" / "coherencia_fao_cfp.svg", bbox_inches="tight")
plt.show()
print(f"Figura guardada: {OUT_DIR}/figuras/coherencia_fao_cfp.png")

## 9. Exportación — CSV + LaTeX

Exportamos los datos FAO procesados en formato FAIR (Findable, Accessible,
Interoperable, Reusable) para reproducibilidad y reutilización.

In [ ]:
# CSV
csv_path = fexp.exportar_fao_csv()
print(f"CSV exportado: {csv_path}")

# Verificar contenido
if csv_path.exists():
    df_check = pd.read_csv(csv_path)
    print(f"  Registros: {len(df_check)} | Columnas: {list(df_check.columns)}")

In [ ]:
# LaTeX
latex_str = fexp.exportar_latex_stocks()
print("Tabla LaTeX generada:")
print("=" * 60)
print(latex_str[:600] + "..." if len(latex_str) > 600 else latex_str)

## 10. Posts LinkedIn — Entrega #04

Generamos los posts para ambos perfiles de la serie:

- **Perfil personal** (Ariel Giamportone): "Argentina pesca más de lo que el mundo recomienda"
  — conectar estado stock FAO con el triángulo CBA→CMP
- **Pesqueros en IA**: "Open data para gobernanza pesquera global: FAO FIRMS + datos locales"

In [ ]:
# Post perfil personal — Entrega #04
post_personal_04 = LinkedInPost(
    numero_entrega=4,
    titulo="Argentina pesca más de lo que el mundo recomienda",
    emoji_tema="🌊",
    hook=(
        "El 34.2% de los stocks mundiales están sobrexplotados (FAO 2022).\n"
        "En Argentina, dos de las cinco especies principales ya están en esa categoría.\n"
        "Y el CFP históricamente aprobó cuotas por encima de la recomendación científica."
    ),
    contexto=(
        "La Entrega #04 de FisheriesAudit ALG cierra el Triángulo de Contexto:\n"
        "• CBA INIDEP → CMP CFP → Captura real (Entregas #01–#03)\n"
        "• + Estado del stock según FAO FIRMS (Área 41, Atlántico Sudoccidental)\n\n"
        "Los datos FAO FishStat muestran que merluza hubbsi y abadejo están evaluados\n"
        "como sobrexplotados en el Atlántico SW. Mientras tanto, Argentina aporta\n"
        "~95% de la captura total de merluza en el Área FAO 41."
    ),
    datos_principales=[
        "Merluza hubbsi (HKP): SOBREXPLOTADO según FAO FIRMS 2024",
        "Abadejo (POA): SOBREXPLOTADO, tendencia declinando (INIDEP + FAO)",
        "Argentina aporta ~95% de captura de merluza en Área FAO 41",
        "2/5 stocks monitoreados en zona de riesgo (40% vs 34.2% promedio mundial)",
        "Merluza negra (TOP) es el caso positivo: regulación CCAMLR efectiva",
    ],
    reflexion=(
        "El triángulo CBA → CMP → FAO cuenta una historia coherente:\n"
        "cuando el CFP aprueba cuotas sistemáticamente sobre la CBA del INIDEP,\n"
        "y el contexto global confirma que el stock está sobrexplotado,\n"
        "la pregunta no es científica. Es de gobernanza.\n\n"
        "¿Quién protege el recurso pesquero cuando los reguladores no lo hacen?"
    ),
    cta=(
        "El análisis completo y el código son abiertos.\n"
        "Serie FisheriesAudit ALG 2026 — 4 entregas, datos públicos, metodología reproducible."
    ),
    hashtags=HASHTAGS_PERSONAL,
    perfil="personal",
    fuentes=[
        "FAO FIRMS 2022–2024 (firms.fao.org)",
        "FAO FishStat (fishstat.fao.org)",
        "INIDEP ITO 36/2024",
        "CFP Actas Públicas",
    ],
)

print("=== POST PERFIL PERSONAL — Entrega #04 ===")
print()
print(post_personal_04.render())

In [ ]:
# Post perfil Pesqueros en IA — Entrega #04
post_ia_04 = LinkedInPost(
    numero_entrega=4,
    titulo="Open data para gobernanza pesquera global: FAO FIRMS + datos locales",
    emoji_tema="📡",
    hook=(
        "FAO publica el estado de 500+ stocks pesqueros en FIRMS.\n"
        "FishStat tiene capturas desde 1950. Todo es open data.\n"
        "¿Por qué los reguladores nacionales no lo integran en sus decisiones?"
    ),
    contexto=(
        "FisheriesAudit ALG integra tres capas de datos abiertos:\n"
        "1. Datos locales: CBA (INIDEP) + CMP (CFP) + Capturas (SAGPyA/SIPA)\n"
        "2. Datos globales: capturas por país y área FAO (FishStat)\n"
        "3. Evaluaciones de stock: FAO FIRMS (Área 41, Atlántico Sudoccidental)\n\n"
        "El resultado es un sistema de alerta temprana que cruza la política\n"
        "pesquera nacional con el estado del recurso a escala global."
    ),
    datos_principales=[
        "FAO FIRMS API: estado de stocks en tiempo real (firms.fao.org/firms/api)",
        "FishStat: series de captura 1950–2022 por país, especie y área FAO",
        "Integración local: SQLite + pandas + scipy para análisis reproducible",
        "Triangulación automática: estado FAO vs ratio CMP/CBA por especie",
        "Stack: Python, FAO FIRMS API, SQLite — 100% open source",
    ],
    reflexion=(
        "La gobernanza pesquera global tiene un problema de integración de datos.\n"
        "Los organismos internacionales (FAO, CCAMLR, IATTC) publican evaluaciones.\n"
        "Los reguladores nacionales toman decisiones. Los datos existen.\n\n"
        "Pero la conexión sistemática entre ambas capas —global y local— es rara.\n"
        "FisheriesAudit ALG es una demostración de que se puede hacer "
        "con herramientas abiertas."
    ),
    cta=(
        "Si estás trabajando en sistemas similares para otras pesquerías "
        "(Chile, Perú, UE, Southeast Asia), conectemos.\n"
        "El código es replicable en cualquier contexto con datos FAO."
    ),
    hashtags=HASHTAGS_PESQUEROS_IA,
    perfil="pesqueros_ia",
    fuentes=[
        "FAO FIRMS API (firms.fao.org/firms/api)",
        "FAO FishStat 2024",
        "GitHub: arielgiamportone/cfp-audit-intelligence",
    ],
)

print("=== POST PESQUEROS EN IA — Entrega #04 ===")
print()
print(post_ia_04.render())

## 11. Metodología y reproducibilidad

### Fuentes de datos

| Fuente | Descripción | Acceso | Período |
|--------|-------------|--------|--------|
| **FAO FishStat** | Capturas globales por especie, país y área FAO | fishstat.fao.org | 2019–2022 |
| **FAO FIRMS** | Estado de stocks pesqueros del Área 41 | firms.fao.org | 2022–2024 |
| **INIDEP ITOs** | Informes Técnicos Operativos (CBA por especie) | marabierto.inidep.edu.ar | 2022–2025 |

Los datos seed verificados (`SEED_DATA_CAPTURAS`, `SEED_STOCK_STATUS`) están documentados
en `src/acquisition/fao_firms_scraper.py` con las referencias exactas de cada registro.

### Tests estadísticos

| Test | Hipótesis | Función |
|------|-----------|--------|
| Kendall tau | Tendencia monótona en capturas argentinas | `test_tendencia_captura_argentina()` |
| Binomial exacto | Proporción sobrexplotación > promedio FAO global | `test_sobrexplotacion_global()` |

### Reproducibilidad

```bash
git clone https://github.com/arielgiamportone/cfp-audit-intelligence
pip install -r requirements.txt
python -c "from src.acquisition.fao_firms_scraper import FAOFIRMSScraper; FAOFIRMSScraper().seed_data()"
jupyter notebook notebooks/FisheriesAudit_ALG_04_contexto_internacional.ipynb
```

### Limitaciones

- Los datos seed cubren 5 especies con datos de captura y 6 con evaluación de stock.
- Las capturas FishStat están disponibles hasta 2022 (último ciclo de publicación FAO).
- La triangulación FAO vs CFP en la Sección 8 es cualitativa; el análisis cuantitativo
  completo requiere el corpus completo del pipeline (492+ actas CFP).
- El test binomial con n≤6 stocks tiene baja potencia estadística.

### Declaración de conflicto de intereses

El autor no tiene vínculos económicos con empresas pesqueras ni con organismos reguladores.
El análisis es descriptivo y no constituye acusación legal.

---

*FisheriesAudit ALG 2026 — Ariel L. Giamportone*  
*Ing. Pesquero | Docente Investigador | Data Scientist*

---

**Entregas de la serie:**
- #01 — Triángulo de Auditoría Pesquera: CBA · CMP · Captura Real
- #02 — Patrones históricos en resoluciones CFP (1998–2025)
- #03 — Red de relaciones empresas-especies-decisiones CFP
- **#04 — Contexto internacional: Argentina en la pesca mundial (FAO FIRMS)** ← actual